# Power BI: build "open this visual" URLs from a visual id

Given a **workspace id**, **report id**, and **visual id**, this notebook downloads the
report's **PBIR** definition from the Fabric REST API, finds every page the visual sits on,
and builds a `shareVisual` deep-link for each one.

**Run order:** run the *Library* cell once, then edit the ids in the *Run* cell and execute it.

**Auth:** inside a Fabric notebook it uses `FabricRestClient` automatically. In a local /
VS Code notebook it falls back to an interactive `azure-identity` sign-in (`pip install
azure-identity requests`). You can also pass `access_token=...` yourself.

Offline sanity check (no auth/network): run `_selftest()` in a cell.


In [ ]:
#!/usr/bin/env python3
"""
Build Power BI "open this visual" deep-link URLs from a workspace + report + visual id.

Given a Fabric / Power BI workspace id, report id, and a visual (visualContainer)
name/id, this script:

  1. Downloads the report's definition via the Fabric REST API
       POST /v1/workspaces/{ws}/reports/{rpt}/getDefinition?format=PBIR
     handling the long-running-operation flow (202 -> poll Location -> GET .../result).
     If the report isn't stored as PBIR, it retries with PBIR-Legacy automatically.
  2. Looks up the workspace name and report name.
  3. Scans the definition for every page whose visuals folder contains that
     visual id -- a visual can be repeated on more than one page -- and reads
     each visual's type and title.
  4. Emits a deep-link URL for each match, in the same shape Power BI's
     "Share > Link to visual" produces:
       https://app.powerbi.com/groups/{ws}/reports/{rpt}/{page}?pbi_source=shareVisual&visual={visual}

Each result row carries: workspace_name, report_name, page (name + display),
visual_id, visual_type, visual_title, and the url. Handy for pandas / CSV:
    import pandas as pd; pd.DataFrame(results)

Runs in a Microsoft Fabric notebook (uses semantic-link's FabricRestClient, which
handles auth for you), in a plain Jupyter / VS Code notebook (falls back to
azure-identity interactive sign-in), or as a CLI script.

Auth resolution order (when you don't pass a token yourself):
  1. sempy.fabric.FabricRestClient       (inside a Fabric notebook)
  2. notebookutils.credentials.getToken  (inside a Fabric notebook, manual path)
  3. azure.identity InteractiveBrowserCredential  (local / VS Code)
The token audience must be https://api.fabric.microsoft.com.

Offline check (no auth, no network):  python powerbi_visual_url.py --selftest
"""

from __future__ import annotations

import base64
import json
import time
from urllib.parse import quote, urlencode

FABRIC_BASE = "https://api.fabric.microsoft.com"
FABRIC_SCOPE = "https://api.fabric.microsoft.com/.default"
DEFAULT_PBI_HOST = "https://app.powerbi.com"


# --------------------------------------------------------------------------- #
# Auth + a tiny HTTP shim that works with either backend
# --------------------------------------------------------------------------- #
_TOKEN_CACHE = {}


def _acquire_token():
    """Obtain a Fabric-audience bearer token when not using FabricRestClient.

    Cached so repeated calls (e.g. the PBIR -> PBIR-Legacy retry) don't prompt
    for sign-in twice.
    """
    if "token" in _TOKEN_CACHE:
        return _TOKEN_CACHE["token"]
    try:
        import notebookutils  # available inside a Fabric notebook
        tok = notebookutils.credentials.getToken("pbi")
    except Exception:
        from azure.identity import InteractiveBrowserCredential
        tok = InteractiveBrowserCredential().get_token(FABRIC_SCOPE).token
    _TOKEN_CACHE["token"] = tok
    return tok


class _FabricHttp:
    """Authenticated GET/POST against Fabric REST, via FabricRestClient (Fabric
    notebook) or plain requests + a bearer token. Both return requests-style
    responses with .status_code, .headers and .json()."""

    def __init__(self, access_token=None):
        self._client = None
        self._requests = None
        self._token = None
        if access_token is None:
            try:
                from sempy.fabric import FabricRestClient
                self._client = FabricRestClient()
            except ImportError:
                self._client = None
        if self._client is None:
            import requests
            self._requests = requests
            self._token = access_token or _acquire_token()

    @staticmethod
    def _rel(url):
        return url.split("api.fabric.microsoft.com")[-1]

    def _abs(self, url):
        return url if url.startswith("http") else f"{FABRIC_BASE}{url}"

    def get(self, url):
        if self._client is not None:
            return self._client.get(self._rel(url))
        return self._requests.get(self._abs(url), headers={"Authorization": f"Bearer {self._token}"})

    def post(self, url):
        if self._client is not None:
            return self._client.post(self._rel(url))
        return self._requests.post(self._abs(url), headers={"Authorization": f"Bearer {self._token}"})


# --------------------------------------------------------------------------- #
# 1. Download the report definition (network) -- PBIR, falling back to legacy
# --------------------------------------------------------------------------- #
class _FormatNotConvertible(Exception):
    """The report can't be exported in the requested definition format."""


def _extract_parts(payload_json):
    return payload_json["definition"]["parts"]


def _complete_lro(get_fn, op_url, retry_after=None):
    """Poll a Fabric long-running operation to completion and return its parts.

    `get_fn(url)` must issue an authenticated GET and return a requests-style
    response. We drive the poll -> /result flow ourselves because semantic-link's
    lro_wait mishandles getDefinition results.
    """
    delay = int(retry_after or 2)
    while True:
        time.sleep(delay)
        poll = get_fn(op_url)
        body = poll.json()
        if isinstance(body, dict) and "definition" in body:
            return _extract_parts(body)
        status = (body or {}).get("status")
        if status == "Succeeded":
            return _extract_parts(get_fn(op_url.rstrip("/") + "/result").json())
        if status in ("Failed", "Undefined"):
            err = (body or {}).get("error") or {}
            if err.get("errorCode") == "Report_Report_FailedToExportReport" or \
                    "cannot be converted" in err.get("message", ""):
                raise _FormatNotConvertible(json.dumps(body))
            raise RuntimeError(f"getDefinition failed: {json.dumps(body)}")
        delay = int(getattr(poll, "headers", {}).get("Retry-After") or delay)


def _fetch_parts_for_format(http, workspace_id, report_id, fmt):
    """Run getDefinition for one format (PBIR or PBIR-Legacy) and return parts."""
    rel = f"/v1/workspaces/{workspace_id}/reports/{report_id}/getDefinition?format={fmt}"
    resp = http.post(rel)
    if resp.status_code == 200:
        return _extract_parts(resp.json())
    if resp.status_code == 202:
        op_url = resp.headers["Location"]
        return _complete_lro(http.get, op_url, resp.headers.get("Retry-After"))
    resp.raise_for_status()
    raise RuntimeError(f"Unexpected status {resp.status_code}")


def _fetch_definition_parts_with(http, workspace_id, report_id):
    """Try PBIR, then fall back to PBIR-Legacy for reports not stored as PBIR."""
    last = None
    for fmt in ("PBIR", "PBIR-Legacy"):
        try:
            return _fetch_parts_for_format(http, workspace_id, report_id, fmt)
        except _FormatNotConvertible as exc:
            last = str(exc)
    raise RuntimeError(
        "getDefinition could not export this report in PBIR or PBIR-Legacy format. "
        f"Last response: {last}"
    )


def fetch_definition_parts(workspace_id, report_id, access_token=None):
    """Return a report's definition parts (PBIR or PBIR-Legacy)."""
    return _fetch_definition_parts_with(_FabricHttp(access_token), workspace_id, report_id)


def get_workspace_name(http, workspace_id):
    try:
        return (http.get(f"/v1/workspaces/{workspace_id}").json() or {}).get("displayName")
    except Exception:
        return None


def get_report_name(http, workspace_id, report_id):
    try:
        return (http.get(f"/v1/workspaces/{workspace_id}/reports/{report_id}").json() or {}).get("displayName")
    except Exception:
        return None


# --------------------------------------------------------------------------- #
# 2. Locate the visual in the definition  (pure -- no network)
# --------------------------------------------------------------------------- #
def _decode(part):
    return base64.b64decode(part["payload"]).decode("utf-8")


def _unquote_literal(value):
    if isinstance(value, str) and len(value) >= 2 and value.startswith("'") and value.endswith("'"):
        return value[1:-1].replace("''", "'")
    return value


def _title_from_objects(objects):
    """Pull a static title string out of a visual 'objects'/'vcObjects' node."""
    if not objects:
        return None
    title = objects.get("title")
    if not title:
        return None
    try:
        lit = title[0]["properties"]["text"]["expr"]["Literal"]["Value"]
        return _unquote_literal(lit)
    except (IndexError, KeyError, TypeError):
        return None


def find_visual_locations(parts, visual_id):
    """
    Return one dict per page on which `visual_id` appears:
        {"page_name", "page_display_name", "visual_type", "visual_title"}
    Handles both PBIR (folder) and PBIR-Legacy (single report.json) definitions.
    """
    is_pbir = any(p["path"].startswith("definition/pages/") for p in parts)
    if is_pbir:
        return _find_in_pbir(parts, visual_id)
    for p in parts:  # PBIR-Legacy: everything lives in one report.json
        if p["path"] == "report.json" or p["path"].endswith("/report.json"):
            return _find_in_legacy(_decode(p), visual_id)
    return []


def _find_in_pbir(parts, visual_id):
    pages = {}
    for p in parts:
        segs = p["path"].split("/")
        if len(segs) == 4 and segs[1] == "pages" and segs[3] == "page.json":
            pages[segs[2]] = json.loads(_decode(p))

    results = []
    for p in parts:
        segs = p["path"].split("/")
        # definition / pages / <pageFolder> / visuals / <visualFolder> / visual.json
        if (
            len(segs) == 6
            and segs[1] == "pages"
            and segs[3] == "visuals"
            and segs[5] in ("visual.json", "visualContainer.json")
        ):
            vj = json.loads(_decode(p))
            name = vj.get("name") or segs[4]
            if name != visual_id:
                continue
            meta = pages.get(segs[2], {})
            v = vj.get("visual") or {}
            vtype = v.get("visualType")
            vtitle = _title_from_objects(v.get("objects"))
            if not vtype and vj.get("visualGroup"):
                vtype = "visualGroup"
                vtitle = (vj.get("visualGroup") or {}).get("displayName")
            results.append(
                {
                    "page_name": meta.get("name", segs[2]),
                    "page_display_name": meta.get("displayName"),
                    "visual_type": vtype,
                    "visual_title": vtitle,
                }
            )
    return results


def _find_in_legacy(report_json_text, visual_id):
    rep = json.loads(report_json_text)
    results = []
    for section in rep.get("sections", []):
        for vc in section.get("visualContainers", []):
            cfg = vc.get("config")
            if not cfg:
                continue
            try:
                cfgj = json.loads(cfg) if isinstance(cfg, str) else cfg
            except (ValueError, TypeError):
                continue
            if cfgj.get("name") != visual_id:
                continue
            sv = cfgj.get("singleVisual") or {}
            vtype = sv.get("visualType")
            vtitle = _title_from_objects(sv.get("vcObjects")) or _title_from_objects(sv.get("objects"))
            if not vtype and cfgj.get("singleVisualGroup"):
                vtype = "visualGroup"
                vtitle = (cfgj.get("singleVisualGroup") or {}).get("displayName")
            results.append(
                {
                    "page_name": section.get("name"),
                    "page_display_name": section.get("displayName"),
                    "visual_type": vtype,
                    "visual_title": vtitle,
                }
            )
    return results


# --------------------------------------------------------------------------- #
# 3. Build the deep-link URL  (pure)
# --------------------------------------------------------------------------- #
def build_visual_url(
    workspace_id,
    report_id,
    page_name,
    visual_id,
    pbi_host=DEFAULT_PBI_HOST,
    ctid=None,
    relative=False,
):
    path = f"/groups/{workspace_id}/reports/{report_id}/{quote(str(page_name), safe='')}"
    params = {}
    if ctid:
        params["ctid"] = ctid
    params["pbi_source"] = "shareVisual"
    params["visual"] = visual_id
    query = urlencode(params)
    return f"{path}?{query}" if relative else f"{pbi_host}{path}?{query}"


# --------------------------------------------------------------------------- #
# 4. Orchestrator
# --------------------------------------------------------------------------- #
def get_visual_urls(
    workspace_id,
    report_id,
    visual_id,
    *,
    access_token=None,
    pbi_host=DEFAULT_PBI_HOST,
    ctid=None,
    parts=None,
    workspace_name=None,
    report_name=None,
):
    """Return one enriched record per page the visual appears on."""
    if parts is None:
        http = _FabricHttp(access_token)
        parts = _fetch_definition_parts_with(http, workspace_id, report_id)
        if workspace_name is None:
            workspace_name = get_workspace_name(http, workspace_id)
        if report_name is None:
            report_name = get_report_name(http, workspace_id, report_id)

    out = []
    for loc in find_visual_locations(parts, visual_id):
        out.append(
            {
                "workspace_id": workspace_id,
                "workspace_name": workspace_name,
                "report_id": report_id,
                "report_name": report_name,
                "page_name": loc["page_name"],
                "page_display_name": loc["page_display_name"],
                "visual_id": visual_id,
                "visual_type": loc.get("visual_type"),
                "visual_title": loc.get("visual_title"),
                "url": build_visual_url(
                    workspace_id, report_id, loc["page_name"], visual_id, pbi_host, ctid
                ),
                "relative_url": build_visual_url(
                    workspace_id, report_id, loc["page_name"], visual_id,
                    pbi_host, ctid, relative=True,
                ),
            }
        )
    return out


def print_results(results):
    if not results:
        print("Visual id not found on any page of this report.")
        return
    first = results[0]
    print(f"Workspace: {first.get('workspace_name') or first.get('workspace_id')}")
    print(f"Report:    {first.get('report_name') or first.get('report_id')}")
    print(f"Visual id: {first.get('visual_id')}")
    print(f"\nFound the visual on {len(results)} page(s):\n")
    for r in results:
        page = r["page_display_name"] or r["page_name"]
        vtype = r.get("visual_type") or "unknown type"
        vtitle = f' "{r["visual_title"]}"' if r.get("visual_title") else ""
        print(f"- Page: {page}  (section id: {r['page_name']})")
        print(f"  Visual:{vtitle} [{vtype}]")
        print(f"  {r['url']}\n")


# =========================================================================== #
# NOTEBOOK USAGE -- fill these in and run the two lines at the bottom.
# =========================================================================== #
WORKSPACE_ID = "7d7e2f04-b1f7-49b4-93c2-1aece6a929d5"
REPORT_ID = "491c4e22-7b10-4d47-9044-c79069681809"
VISUAL_ID = "c83ed47768b1aeb380d0"
CTID = None  # optional tenant id, e.g. "d2d5283f-21bf-4fb9-bfa1-1e91215840c1"; omit if not needed

# In a notebook cell, run:
#     results = get_visual_urls(WORKSPACE_ID, REPORT_ID, VISUAL_ID, ctid=CTID)
#     print_results(results)
#     # results is a list of dicts -> pandas: import pandas as pd; pd.DataFrame(results)


# --------------------------------------------------------------------------- #
# Offline self-test  (no auth / no network)
# --------------------------------------------------------------------------- #
def _selftest():
    def part(path, obj):
        payload = base64.b64encode(json.dumps(obj).encode()).decode()
        return {"path": path, "payload": payload, "payloadType": "InlineBase64"}

    vid = "c83ed47768b1aeb380d0"
    page1 = "ReportSection942c376307e8dc6a520a"
    page2 = "ReportSectionAAAA1111bbbb2222cccc"
    ws = "7d7e2f04-b1f7-49b4-93c2-1aece6a929d5"
    rpt = "491c4e22-7b10-4d47-9044-c79069681809"

    parts = [
        part("definition/report.json", {"$schema": "x"}),
        part(f"definition/pages/{page1}/page.json", {"name": page1, "displayName": "Overview"}),
        part(
            f"definition/pages/{page1}/visuals/{vid}/visual.json",
            {
                "name": vid,
                "visual": {
                    "visualType": "clusteredColumnChart",
                    "objects": {
                        "title": [
                            {"properties": {"text": {"expr": {"Literal": {"Value": "'My Title'"}}}}}
                        ]
                    },
                },
            },
        ),
        # same visual id reused on a second page -> should yield a second URL
        part(f"definition/pages/{page2}/page.json", {"name": page2, "displayName": "Detail"}),
        part(f"definition/pages/{page2}/visuals/{vid}/visual.json", {"name": vid}),
        # decoy visual that must NOT match
        part(f"definition/pages/{page1}/visuals/deadbeef/visual.json", {"name": "deadbeef"}),
    ]

    results = get_visual_urls(
        ws, rpt, vid, ctid="d2d5283f-21bf-4fb9-bfa1-1e91215840c1", parts=parts
    )
    assert len(results) == 2, f"expected 2 pages, got {len(results)}"
    assert results[0]["visual_type"] == "clusteredColumnChart", results[0]
    assert results[0]["visual_title"] == "My Title", results[0]
    expected = (
        "https://app.powerbi.com/groups/7d7e2f04-b1f7-49b4-93c2-1aece6a929d5/"
        "reports/491c4e22-7b10-4d47-9044-c79069681809/"
        "ReportSection942c376307e8dc6a520a?"
        "ctid=d2d5283f-21bf-4fb9-bfa1-1e91215840c1&pbi_source=shareVisual&"
        "visual=c83ed47768b1aeb380d0"
    )
    assert results[0]["url"] == expected, f"URL mismatch:\n got: {results[0]['url']}\n exp: {expected}"

    # PBIR-Legacy fallback + title/type extraction from a legacy config
    legacy = [
        part(
            "report.json",
            {
                "sections": [
                    {
                        "name": page1,
                        "displayName": "Overview",
                        "visualContainers": [
                            {
                                "config": json.dumps(
                                    {
                                        "name": vid,
                                        "singleVisual": {
                                            "visualType": "pivotTable",
                                            "vcObjects": {
                                                "title": [
                                                    {"properties": {"text": {"expr": {"Literal": {"Value": "'Legacy Title'"}}}}}
                                                ]
                                            },
                                        },
                                    }
                                )
                            }
                        ],
                    }
                ]
            },
        )
    ]
    legacy_results = get_visual_urls(ws, rpt, vid, parts=legacy)
    assert len(legacy_results) == 1, f"legacy expected 1, got {len(legacy_results)}"
    assert legacy_results[0]["visual_type"] == "pivotTable", legacy_results[0]
    assert legacy_results[0]["visual_title"] == "Legacy Title", legacy_results[0]

    # ---- LRO + PBIR -> PBIR-Legacy fallback (stubbed; no network) ----
    class _Resp:
        def __init__(self, body):
            self._b = body
            self.headers = {}

        def json(self):
            return self._b

    real_sleep = time.sleep
    time.sleep = lambda *a, **k: None
    try:
        fail_body = {
            "status": "Failed",
            "error": {"errorCode": "Report_Report_FailedToExportReport",
                      "message": "... cannot be converted ..."},
        }
        try:
            _complete_lro(lambda u: _Resp(fail_body), "https://api.fabric.microsoft.com/v1/operations/x")
            raise AssertionError("expected _FormatNotConvertible")
        except _FormatNotConvertible:
            pass

        def _get_ok(u):
            if u.endswith("/result"):
                return _Resp({"definition": {"parts": [{"path": "report.json"}]}})
            return _Resp({"status": "Succeeded"})

        assert _complete_lro(_get_ok, "https://api.fabric.microsoft.com/v1/operations/x") == [
            {"path": "report.json"}
        ]

        global _fetch_parts_for_format
        original = _fetch_parts_for_format
        calls = []

        def _fake(http, ws_, rpt_, fmt):
            calls.append(fmt)
            if fmt == "PBIR":
                raise _FormatNotConvertible("stored as legacy")
            return [{"path": "report.json"}]

        _fetch_parts_for_format = _fake
        try:
            assert _fetch_definition_parts_with(None, "w", "r") == [{"path": "report.json"}]
        finally:
            _fetch_parts_for_format = original
        assert calls == ["PBIR", "PBIR-Legacy"], calls
    finally:
        time.sleep = real_sleep

    print("selftest OK")
    print(expected)


## Run

In [ ]:
# --- Edit these, then run ---
WORKSPACE_ID = "7d7e2f04-b1f7-49b4-93c2-1aece6a929d5"
REPORT_ID    = "491c4e22-7b10-4d47-9044-c79069681809"
VISUAL_ID    = "c83ed47768b1aeb380d0"
CTID         = None   # optional tenant id, e.g. "d2d5283f-21bf-4fb9-bfa1-1e91215840c1"

results = get_visual_urls(WORKSPACE_ID, REPORT_ID, VISUAL_ID, ctid=CTID)
print_results(results)
results
